# Rotating Snowflake Service Account Credentials with AWS Secrets Manager

> **No support provided.** This code is for reference only. Review, test, and modify before any production use.
> All object names in this workbook (users, roles, policies) are **examples**. Replace them with your own names before applying to production service accounts.

**Pair-programmed by:** SE Community + Cortex Code

## Why This Matters

Service accounts (`TYPE=SERVICE`) are the backbone of automated Snowflake workloads: ETL pipelines, CI/CD deployments, API integrations, and application backends. When their credentials are static, a single leak exposes every system that shares them.

**The pressure is increasing:**

- Snowflake is [deprecating single-factor password authentication](https://docs.snowflake.com/en/user-guide/security-mfa-rollout) -- service accounts must move to key-pair or PAT-based auth
- Compliance frameworks (SOC 2, PCI DSS, HIPAA) require regular credential rotation
- A compromised service account key that never rotates gives an attacker indefinite access

This workbook covers two automated rotation patterns using AWS Secrets Manager so credentials rotate on schedule with zero downtime.

## What This Workbook Covers

| Pattern | Auth Method | Rotation Mechanism | Best For |
|---------|-------------|-------------------|----------|
| Pattern 1 | RSA Key-Pair | AWS Secrets Manager native `SnowflakeKeyPairAuthentication` | Drivers, connectors, JWT-based API calls |
| Pattern 2 | Programmatic Access Token (PAT) | Custom Lambda + `ALTER USER ROTATE PAT` | REST API calls, third-party tools accepting bearer tokens |

Both patterns store secrets in AWS Secrets Manager and rotate them automatically via Lambda.

Architecture diagrams are in [`diagrams.md`](https://github.com/sfc-gh-miwhitaker/sfe-public/blob/main/tool-secrets-rotation-aws/diagrams.md) on GitHub (Mermaid does not render inside Snowflake Notebooks).

## Architecture Overview

```
┌─────────────────────────────────────────────────────────────────────┐
│                              AWS                                    │
│  ┌──────────────────┐   ┌───────────────────┐   ┌───────────────┐ │
│  │ Secrets Manager  │──▶│ Key-Pair Rotation  │──▶│  Snowflake    │ │
│  │ (stores keys &   │   │ Lambda (managed)   │   │  Service User │ │
│  │  PATs)           │──▶│ PAT Rotation       │──▶│  ALTER USER   │ │
│  └────────┬─────────┘   │ Lambda (custom)    │   └───────────────┘ │
│           │              └───────────────────┘                      │
│           │                                                         │
│  ┌────────▼─────────┐                                              │
│  │  Applications    │   GetSecretValue at runtime                  │
│  │  (ETL, API, CI)  │──────────────────────────▶ Snowflake        │
│  └──────────────────┘                                              │
└─────────────────────────────────────────────────────────────────────┘
```

- **Pattern 1:** AWS Secrets Manager natively generates and rotates RSA key pairs, pushing the public key to Snowflake via `ALTER USER SET RSA_PUBLIC_KEY`.
- **Pattern 2:** A custom Lambda authenticates via key-pair (from Pattern 1), runs `ALTER USER ROTATE PAT`, captures the new token, and stores it in Secrets Manager.
- **Applications** fetch the current credential from Secrets Manager at runtime -- never from disk.

---
# Section 1: Create Example Service Account

We create a purpose-built example service user with all the supporting infrastructure: a least-privilege role, network policy, authentication policy, and a rotator role.

**All object names use the `SFE_` prefix** to make them obviously identifiable as examples. Replace these with your own naming conventions for production use.

In [ ]:
-- Create the example service user
USE ROLE USERADMIN;

CREATE USER IF NOT EXISTS SFE_SVC_ROTATION_EXAMPLE
  TYPE = SERVICE
  DEFAULT_ROLE = 'SFE_SVC_ROTATION_ROLE'
  DEFAULT_WAREHOUSE = 'SFE_TOOLS_WH'
  COMMENT = 'TOOL: Example service account for rotation workbook (Expires: 2026-04-05)';

In [ ]:
-- Create and grant the service role
USE ROLE SECURITYADMIN;

CREATE ROLE IF NOT EXISTS SFE_SVC_ROTATION_ROLE
  COMMENT = 'TOOL: Least-privilege role for SFE_SVC_ROTATION_EXAMPLE (Expires: 2026-04-05)';

GRANT ROLE SFE_SVC_ROTATION_ROLE TO USER SFE_SVC_ROTATION_EXAMPLE;
GRANT USAGE ON WAREHOUSE SFE_TOOLS_WH TO ROLE SFE_SVC_ROTATION_ROLE;
GRANT USAGE ON DATABASE SNOWFLAKE_EXAMPLE TO ROLE SFE_SVC_ROTATION_ROLE;
GRANT USAGE ON SCHEMA SNOWFLAKE_EXAMPLE.SECRETS_ROTATION TO ROLE SFE_SVC_ROTATION_ROLE;

### Network Policy

Service users (`TYPE=SERVICE`) **must** be subject to a network policy to generate and use PATs. Even for key-pair-only setups, a network policy limits where the credentials can be used.

The example below allows common private CIDR ranges. In production, restrict this to your specific VPC/Lambda IP ranges.

In [ ]:
USE ROLE SYSADMIN;

CREATE NETWORK RULE IF NOT EXISTS SNOWFLAKE_EXAMPLE.SECRETS_ROTATION.SFE_SVC_ROTATION_NETWORK_RULE
  TYPE = IPV4
  VALUE_LIST = ('10.0.0.0/8', '172.16.0.0/12', '192.168.0.0/16')
  MODE = INGRESS
  COMMENT = 'TOOL: CIDR ranges for rotation example (Expires: 2026-04-05)';

USE ROLE SECURITYADMIN;

CREATE NETWORK POLICY IF NOT EXISTS SFE_SVC_ROTATION_NETWORK_POLICY
  ALLOWED_NETWORK_RULE_LIST = ('SNOWFLAKE_EXAMPLE.SECRETS_ROTATION.SFE_SVC_ROTATION_NETWORK_RULE')
  COMMENT = 'TOOL: Restricts SFE_SVC_ROTATION_EXAMPLE to allowed CIDRs (Expires: 2026-04-05)';

ALTER USER SFE_SVC_ROTATION_EXAMPLE SET NETWORK_POLICY = 'SFE_SVC_ROTATION_NETWORK_POLICY';

In [ ]:
-- Create authentication policy: allow only key-pair and PAT auth methods
USE ROLE SYSADMIN;

CREATE AUTHENTICATION POLICY IF NOT EXISTS SNOWFLAKE_EXAMPLE.SECRETS_ROTATION.SFE_SVC_ROTATION_AUTH_POLICY
  AUTHENTICATION_METHODS = ('PROGRAMMATIC_ACCESS_TOKEN', 'KEYPAIR')
  PAT_POLICY = (
    DEFAULT_EXPIRY_IN_DAYS = 30,
    MAX_EXPIRY_IN_DAYS = 90,
    NETWORK_POLICY_EVALUATION = ENFORCED_REQUIRED
  )
  COMMENT = 'TOOL: Auth policy for rotation example - key-pair + PAT only (Expires: 2026-04-05)';

USE ROLE SECURITYADMIN;

ALTER USER SFE_SVC_ROTATION_EXAMPLE
  SET AUTHENTICATION POLICY SNOWFLAKE_EXAMPLE.SECRETS_ROTATION.SFE_SVC_ROTATION_AUTH_POLICY;

In [ ]:
-- Create the rotator role (used by Lambda to perform rotation)
USE ROLE SECURITYADMIN;

CREATE ROLE IF NOT EXISTS SFE_SVC_ROTATION_ROTATOR_ROLE
  COMMENT = 'TOOL: Role used by Lambda to rotate credentials for SFE_SVC_ROTATION_EXAMPLE (Expires: 2026-04-05)';

GRANT MODIFY PROGRAMMATIC AUTHENTICATION METHODS ON USER SFE_SVC_ROTATION_EXAMPLE
  TO ROLE SFE_SVC_ROTATION_ROTATOR_ROLE;

---
# Section 2: Pattern 1 -- Key-Pair Rotation

AWS Secrets Manager has native support for Snowflake RSA key-pair authentication via the `SnowflakeKeyPairAuthentication` secret type. This is the recommended approach when your applications connect via Snowflake drivers, connectors, or JWT-based API calls.

### How It Works

1. **Secrets Manager triggers rotation** on the configured schedule (e.g., every 30 days)
2. **Lambda generates a new RSA key pair** and stores the private key as `AWSPENDING`
3. **Lambda pushes the public key to Snowflake** via `ALTER USER SET RSA_PUBLIC_KEY_2` (using the alternate slot)
4. **Lambda tests the connection** with the new key pair
5. **Lambda promotes `AWSPENDING` to `AWSCURRENT`** -- applications now receive the new key
6. On the next rotation, the cycle reverses (slot 2 becomes active, slot 1 gets the new key)

The two public key slots (`RSA_PUBLIC_KEY` and `RSA_PUBLIC_KEY_2`) ensure **zero downtime** during rotation. Both keys remain valid until the old one is explicitly unset.

### Step 1: Generate Key Pair (Local Terminal)

Run these commands in your **local terminal** (not in this notebook):

```bash
# Generate unencrypted private key
openssl genrsa 2048 | openssl pkcs8 -topk8 -inform PEM -out rsa_key.p8 -nocrypt

# Generate public key
openssl rsa -in rsa_key.p8 -pubout -out rsa_key.pub

# Display the public key (copy this for the next cell)
cat rsa_key.pub
```

Strip the `-----BEGIN PUBLIC KEY-----` and `-----END PUBLIC KEY-----` lines, then paste the base64 content into the next SQL cell.

In [ ]:
-- Assign the public key to the example service user
-- Replace the placeholder below with your actual public key (no BEGIN/END lines)
USE ROLE USERADMIN;

ALTER USER SFE_SVC_ROTATION_EXAMPLE
  SET RSA_PUBLIC_KEY = 'MIIBIjANBgkqhkiG9w0BAQEFAAOCAQ8AMIIBCgKCAQEA...<PASTE YOUR KEY HERE>...IDAQAB';

In [ ]:
-- Verify the key-pair fingerprint
-- Compare the output with: openssl rsa -pubin -in rsa_key.pub -outform DER | openssl dgst -sha256 -binary | openssl enc -base64
DESC USER SFE_SVC_ROTATION_EXAMPLE;

In [ ]:
-- Extract just the fingerprints for easy comparison
SELECT
    "property",
    "value"
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
WHERE "property" IN ('RSA_PUBLIC_KEY_FP', 'RSA_PUBLIC_KEY_2_FP');

### Step 2: Create the Secret in AWS Secrets Manager

Run in your **local terminal** or AWS Console:

```bash
aws secretsmanager create-secret \
  --name "snowflake/svc-rotation-example/keypair" \
  --description "Snowflake key-pair for SFE_SVC_ROTATION_EXAMPLE" \
  --secret-string '{
    "account": "myorg-myaccount",
    "user": "SFE_SVC_ROTATION_EXAMPLE",
    "privateKey": "'"$(cat rsa_key.p8)"'",
    "publicKey": "'"$(cat rsa_key.pub)"'"
  }'
```

Or in the AWS Console: **Secrets Manager > Store a new secret > Snowflake (RSA key pair)**

### Step 3: Configure Automatic Rotation

```bash
aws secretsmanager rotate-secret \
  --secret-id "snowflake/svc-rotation-example/keypair" \
  --rotation-rules '{"AutomaticallyAfterDays": 30}' \
  --rotate-immediately
```

AWS provides a managed rotation Lambda for `SnowflakeKeyPairAuthentication`. The Lambda needs:
- **Network access** to your Snowflake account (VPC endpoint or NAT gateway)
- **IAM permissions** for `secretsmanager:GetSecretValue`, `PutSecretValue`, `UpdateSecretVersionStage`, `DescribeSecret`

### Application Code Pattern (Python)

Applications fetch the private key at runtime -- never store it on disk:

```python
import boto3
import json
import snowflake.connector
from cryptography.hazmat.primitives import serialization


def get_snowflake_connection():
    sm = boto3.client("secretsmanager")
    response = sm.get_secret_value(SecretId="snowflake/svc-rotation-example/keypair")
    secret = json.loads(response["SecretString"])

    no_passphrase = None
    private_key = serialization.load_pem_private_key(
        secret["privateKey"].encode(), no_passphrase
    )
    pkb = private_key.private_bytes(
        encoding=serialization.Encoding.DER,
        format=serialization.PrivateFormat.PKCS8,
        encryption_algorithm=serialization.NoEncryption(),
    )

    return snowflake.connector.connect(
        account=secret["account"],
        user=secret["user"],
        private_key=pkb,
    )
```

### IAM Policy for the Rotation Lambda

```json
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Action": [
        "secretsmanager:GetSecretValue",
        "secretsmanager:PutSecretValue",
        "secretsmanager:UpdateSecretVersionStage",
        "secretsmanager:DescribeSecret"
      ],
      "Resource": "arn:aws:secretsmanager:us-east-1:123456789012:secret:snowflake/svc-rotation-example/keypair-*"
    }
  ]
}
```

---
# Section 3: Pattern 2 -- PAT Rotation

Programmatic Access Tokens (PATs) are ideal for REST API authentication (SQL API, Cortex Agents, Snowpark Container Services). Unlike key-pair auth, PAT rotation requires a **custom** Lambda because:

- The new token secret **only appears in the output** of `ALTER USER ROTATE PAT` -- it cannot be retrieved later
- PAT rotation **cannot be performed from a PAT-authenticated session** -- the Lambda must authenticate via key-pair

This pattern layers on top of Pattern 1: the Lambda uses a key-pair secret to authenticate to Snowflake, then rotates the PAT and stores the new token in a separate secret.

In [ ]:
-- Create the initial PAT for the example service user
USE ROLE SECURITYADMIN;

ALTER USER SFE_SVC_ROTATION_EXAMPLE
  ADD PROGRAMMATIC ACCESS TOKEN SFE_ROTATION_EXAMPLE_PAT
  DAYS_TO_EXPIRY = 30
  ROLE_RESTRICTION = 'SFE_SVC_ROTATION_ROLE'
  COMMENT = 'TOOL: Example PAT for rotation workbook (rotated by AWS Secrets Manager)';

-- IMPORTANT: Copy the token_secret from the output immediately!
-- This is the ONLY time the secret will be displayed.

In [ ]:
-- Verify the PAT was created
SHOW USER PROGRAMMATIC ACCESS TOKENS FOR USER SFE_SVC_ROTATION_EXAMPLE;

In [ ]:
-- Rotate the PAT (live rotation!)
-- The old token remains valid for 24 hours (grace period for in-flight requests)
USE ROLE SECURITYADMIN;

ALTER USER SFE_SVC_ROTATION_EXAMPLE
  ROTATE PROGRAMMATIC ACCESS TOKEN SFE_ROTATION_EXAMPLE_PAT
  EXPIRE_ROTATED_TOKEN_AFTER_HOURS = 24;

-- Copy the new token_secret from the output!
-- The output also shows the rotated_token_name (the old token object).

In [ ]:
-- Verify: you should now see BOTH the new token and the old rotated token
SHOW USER PROGRAMMATIC ACCESS TOKENS FOR USER SFE_SVC_ROTATION_EXAMPLE;

### Store PAT in AWS Secrets Manager

Run in your **local terminal**:

```bash
aws secretsmanager create-secret \
  --name "snowflake/svc-rotation-example/pat" \
  --description "Snowflake PAT for SFE_SVC_ROTATION_EXAMPLE" \
  --secret-string '{
    "account": "myorg-myaccount",
    "user": "SFE_SVC_ROTATION_EXAMPLE",
    "pat_name": "SFE_ROTATION_EXAMPLE_PAT",
    "token_secret": "<paste-token-here>"
  }'
```

### Lambda Rotation Function (Custom)

The Lambda follows the standard 4-step Secrets Manager rotation lifecycle. It authenticates to Snowflake via key-pair (Pattern 1), then rotates the PAT.

```python
import boto3
import json
import snowflake.connector
from cryptography.hazmat.primitives import serialization

sm = boto3.client("secretsmanager")
KEYPAIR_SECRET_ID = "snowflake/svc-rotation-example/keypair"


def lambda_handler(event, context):
    step = event["Step"]
    secret_id = event["SecretId"]
    token = event["ClientRequestToken"]

    if step == "createSecret":
        create_secret(secret_id, token)
    elif step == "setSecret":
        pass  # Snowflake handles this atomically in ROTATE PAT
    elif step == "testSecret":
        test_secret(secret_id, token)
    elif step == "finishSecret":
        finish_secret(secret_id, token)


def create_secret(secret_id, token):
    current = json.loads(
        sm.get_secret_value(SecretId=secret_id, VersionStage="AWSCURRENT")["SecretString"]
    )
    conn = _get_keypair_connection(current["account"])
    try:
        cur = conn.cursor()
        cur.execute(
            f"ALTER USER {current['user']} "
            f"ROTATE PROGRAMMATIC ACCESS TOKEN {current['pat_name']} "
            f"EXPIRE_ROTATED_TOKEN_AFTER_HOURS = 24"
        )
        new_token = cur.fetchone()[1]  # token_secret column
    finally:
        conn.close()

    sm.put_secret_value(
        SecretId=secret_id,
        ClientRequestToken=token,
        SecretString=json.dumps({**current, "token_secret": new_token}),
        VersionStages=["AWSPENDING"],
    )


def test_secret(secret_id, token):
    pending = json.loads(
        sm.get_secret_value(SecretId=secret_id, VersionId=token, VersionStage="AWSPENDING")["SecretString"]
    )
    import requests
    resp = requests.get(
        f"https://{pending['account']}.snowflakecomputing.com/api/v2/databases",
        headers={"Authorization": f"Bearer {pending['token_secret']}"},
        timeout=10,
    )
    if resp.status_code != 200:
        raise Exception(f"PAT test failed: HTTP {resp.status_code}")


def finish_secret(secret_id, token):
    metadata = sm.describe_secret(SecretId=secret_id)
    for vid, stages in metadata["VersionIdsToStages"].items():
        if "AWSCURRENT" in stages and vid != token:
            sm.update_secret_version_stage(
                SecretId=secret_id, VersionStage="AWSCURRENT",
                MoveToVersionId=token, RemoveFromVersionId=vid,
            )
            sm.update_secret_version_stage(
                SecretId=secret_id, VersionStage="AWSPENDING",
                RemoveFromVersionId=token,
            )
            break


def _get_keypair_connection(account):
    kp = json.loads(sm.get_secret_value(SecretId=KEYPAIR_SECRET_ID)["SecretString"])
    no_passphrase = None
    pk = serialization.load_pem_private_key(kp["privateKey"].encode(), no_passphrase)
    return snowflake.connector.connect(
        account=account, user=kp["user"],
        private_key=pk.private_bytes(
            serialization.Encoding.DER,
            serialization.PrivateFormat.PKCS8,
            serialization.NoEncryption(),
        ),
    )
```

Deploy and enable rotation:

```bash
aws secretsmanager rotate-secret \
  --secret-id "snowflake/svc-rotation-example/pat" \
  --rotation-lambda-arn "arn:aws:lambda:us-east-1:123456789012:function:rotate-snowflake-pat" \
  --rotation-rules '{"AutomaticallyAfterDays": 14}' \
  --rotate-immediately
```

### IAM Policy (PAT Rotation Lambda)

Needs access to **both** secrets (key-pair for auth, PAT for rotation):

```json
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Action": ["secretsmanager:GetSecretValue"],
      "Resource": "arn:aws:secretsmanager:us-east-1:123456789012:secret:snowflake/svc-rotation-example/keypair-*"
    },
    {
      "Effect": "Allow",
      "Action": ["secretsmanager:GetSecretValue", "secretsmanager:PutSecretValue", "secretsmanager:UpdateSecretVersionStage", "secretsmanager:DescribeSecret"],
      "Resource": "arn:aws:secretsmanager:us-east-1:123456789012:secret:snowflake/svc-rotation-example/pat-*"
    }
  ]
}
```

---
# Section 4: Monitoring & Alerting

These queries help you monitor credential health across your account. Two data sources:

- **`SNOWFLAKE.ACCOUNT_USAGE.CREDENTIALS`** -- Account-wide PAT inventory (up to **2-hour latency**)
- **`SHOW USER PROGRAMMATIC ACCESS TOKENS`** -- Per-user PAT listing (**real-time**, no latency)

Run the cells below with `ACCOUNTADMIN` or a role with access to `SNOWFLAKE.ACCOUNT_USAGE` views.

In [ ]:
-- 1. PAT INVENTORY: All programmatic access tokens across the account
USE ROLE ACCOUNTADMIN;

SELECT
    credential_id,
    name                                                        AS pat_name,
    user_name,
    status,
    expiration_date,
    DATEDIFF('day', CURRENT_TIMESTAMP(), expiration_date)       AS days_until_expiry,
    last_used_on,
    created_on,
    created_by,
    additional_details:"ROLE_RESTRICTION"::ARRAY                AS role_restriction,
    additional_details:"ROTATED_TO"::VARCHAR                    AS rotated_to
FROM SNOWFLAKE.ACCOUNT_USAGE.CREDENTIALS
WHERE type = 'PAT'
ORDER BY expiration_date ASC;

In [ ]:
-- 2. EXPIRATION ALERTING: PATs expiring within the next 7 days
SELECT
    name                                                        AS pat_name,
    user_name,
    status,
    expiration_date,
    DATEDIFF('day', CURRENT_TIMESTAMP(), expiration_date)       AS days_until_expiry,
    last_used_on,
    CASE
        WHEN DATEDIFF('day', CURRENT_TIMESTAMP(), expiration_date) <= 0
        THEN 'EXPIRED'
        WHEN DATEDIFF('day', CURRENT_TIMESTAMP(), expiration_date) <= 3
        THEN 'CRITICAL'
        ELSE 'WARNING'
    END                                                         AS urgency
FROM SNOWFLAKE.ACCOUNT_USAGE.CREDENTIALS
WHERE type = 'PAT'
  AND status = 'ACTIVE'
  AND expiration_date BETWEEN CURRENT_TIMESTAMP() AND DATEADD('day', 7, CURRENT_TIMESTAMP())
ORDER BY expiration_date ASC;

In [ ]:
-- 3. STALE PATS: Active tokens that haven't been used in 30+ days
SELECT
    name                                                        AS pat_name,
    user_name,
    status,
    expiration_date,
    last_used_on,
    DATEDIFF('day', last_used_on, CURRENT_TIMESTAMP())          AS days_since_last_use,
    created_on,
    created_by
FROM SNOWFLAKE.ACCOUNT_USAGE.CREDENTIALS
WHERE type = 'PAT'
  AND status = 'ACTIVE'
  AND (last_used_on IS NULL OR last_used_on < DATEADD('day', -30, CURRENT_TIMESTAMP()))
ORDER BY last_used_on ASC NULLS FIRST;

In [ ]:
-- 4. ROTATED TOKEN CLEANUP: Tokens left behind by rotation
--    Rotated tokens count toward the 15-per-user limit
SELECT
    name                                                        AS pat_name,
    user_name,
    status,
    expiration_date,
    additional_details:"ROTATED_TO"::VARCHAR                    AS rotated_to
FROM SNOWFLAKE.ACCOUNT_USAGE.CREDENTIALS
WHERE type = 'PAT'
  AND additional_details:"ROTATED_TO" IS NOT NULL
  AND status IN ('ACTIVE', 'EXPIRED')
ORDER BY user_name, expiration_date ASC;

In [ ]:
-- 5. PAT COUNT PER USER: Ensure no user is approaching the 15-token limit
SELECT
    user_name,
    COUNT(*)                                                    AS total_pats,
    COUNT_IF(status = 'ACTIVE')                                 AS active_pats,
    COUNT_IF(status = 'DISABLED')                               AS disabled_pats,
    COUNT_IF(status = 'EXPIRED')                                AS expired_pats,
    15 - COUNT_IF(status IN ('ACTIVE', 'DISABLED'))             AS remaining_slots
FROM SNOWFLAKE.ACCOUNT_USAGE.CREDENTIALS
WHERE type = 'PAT'
GROUP BY user_name
ORDER BY total_pats DESC;

In [ ]:
-- 6. KEY-PAIR FINGERPRINT VERIFICATION for example user
DESC USER SFE_SVC_ROTATION_EXAMPLE;

In [ ]:
-- Extract just the fingerprints
SELECT
    "property",
    "value"
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
WHERE "property" IN ('RSA_PUBLIC_KEY_FP', 'RSA_PUBLIC_KEY_2_FP');

In [ ]:
-- 7. LOGIN AUDIT TRAIL: Sessions authenticated via PAT (last 30 days)
SELECT
    lh.event_timestamp,
    lh.user_name,
    cred.name                                                   AS pat_name,
    lh.client_ip,
    lh.reported_client_type,
    lh.is_success,
    lh.error_code,
    lh.error_message
FROM SNOWFLAKE.ACCOUNT_USAGE.LOGIN_HISTORY lh
  JOIN SNOWFLAKE.ACCOUNT_USAGE.CREDENTIALS cred
    ON lh.first_authentication_factor_id = cred.credential_id
WHERE lh.first_authentication_factor = 'PROGRAMMATIC_ACCESS_TOKEN'
  AND lh.event_timestamp >= DATEADD('day', -30, CURRENT_TIMESTAMP())
ORDER BY lh.event_timestamp DESC;

In [ ]:
-- 8. FAILED PAT LOGINS: Detect potential credential misuse or rotation issues
SELECT
    lh.event_timestamp,
    lh.user_name,
    cred.name                                                   AS pat_name,
    lh.client_ip,
    lh.error_code,
    lh.error_message
FROM SNOWFLAKE.ACCOUNT_USAGE.LOGIN_HISTORY lh
  JOIN SNOWFLAKE.ACCOUNT_USAGE.CREDENTIALS cred
    ON lh.first_authentication_factor_id = cred.credential_id
WHERE lh.first_authentication_factor = 'PROGRAMMATIC_ACCESS_TOKEN'
  AND lh.is_success = 'NO'
  AND lh.event_timestamp >= DATEADD('day', -7, CURRENT_TIMESTAMP())
ORDER BY lh.event_timestamp DESC;

In [ ]:
-- 9. KEY-PAIR LOGIN AUDIT: Sessions authenticated via key-pair (last 30 days)
SELECT
    event_timestamp,
    user_name,
    client_ip,
    reported_client_type,
    first_authentication_factor,
    is_success,
    error_code,
    error_message
FROM SNOWFLAKE.ACCOUNT_USAGE.LOGIN_HISTORY
WHERE first_authentication_factor = 'RSA_KEYPAIR'
  AND event_timestamp >= DATEADD('day', -30, CURRENT_TIMESTAMP())
ORDER BY event_timestamp DESC;

In [ ]:
-- 10. PER-USER PAT STATUS (real-time, no latency)
SHOW USER PROGRAMMATIC ACCESS TOKENS FOR USER SFE_SVC_ROTATION_EXAMPLE;

### Decode a Leaked Token

If a PAT secret is found outside its intended location, use `SYSTEM$DECODE_PAT` to identify the user and token name:

```sql
SELECT SYSTEM$DECODE_PAT('<paste-leaked-token-here>');
```

The output includes `STATE`, `PAT_NAME`, and `USER_NAME` -- enough to revoke the token immediately.

---
# Security Checklist

- [ ] Service accounts use `TYPE=SERVICE` (not `PERSON` or `LEGACY_SERVICE`)
- [ ] Every service user is subject to a network policy restricting access to known infrastructure CIDRs
- [ ] Authentication policy explicitly lists allowed methods (`KEYPAIR`, `PROGRAMMATIC_ACCESS_TOKEN`)
- [ ] PATs have `ROLE_RESTRICTION` set to a least-privilege role
- [ ] PAT `MAX_EXPIRY_IN_DAYS` is set to a reasonable ceiling (e.g., 90 days)
- [ ] Key-pair rotation is scheduled (recommended: every 30 days)
- [ ] PAT rotation is scheduled (recommended: every 14 days)
- [ ] Lambda functions run inside a VPC with access to Snowflake (VPC endpoint or NAT gateway)
- [ ] Private keys are never stored on disk -- fetched from Secrets Manager at runtime
- [ ] Rotation Lambda IAM policies follow least privilege (scoped to specific secret ARNs)
- [ ] CloudTrail logging is enabled for Secrets Manager API calls
- [ ] Snowflake `LOGIN_HISTORY` and `CREDENTIALS` views are monitored for anomalies
- [ ] Old rotated PAT tokens are cleaned up (they count toward the 15-token-per-user limit)
- [ ] `SYSTEM$DECODE_PAT` is available for incident response if a token is leaked

## Key Limitations and Gotchas

| Limitation | Impact | Mitigation |
|-----------|--------|------------|
| Cannot rotate PAT from a PAT-authenticated session | Lambda must use key-pair auth | Pattern 2 depends on Pattern 1 being in place |
| PAT `token_secret` only appears once | Miss it and you must create a new PAT | Lambda captures it in the same transaction |
| Max 15 PATs per user (includes disabled) | Rotated tokens accumulate | Clean up old `_ROTATED_` tokens after grace period |
| `CREDENTIALS` view has up to 2-hour latency | Near-real-time monitoring not possible via this view | Use `SHOW USER PROGRAMMATIC ACCESS TOKENS` for immediate status |
| Expired PATs auto-delete after 7 days | Cannot audit expired tokens after 7 days | Export to external logging before expiry |
| Service users require network policy for PATs | Lambda must originate from allowed IP range | Run Lambda in VPC with NAT gateway in the allowed CIDR |

---
# References

### Snowflake Documentation

- [Key-Pair Authentication and Key-Pair Rotation](https://docs.snowflake.com/en/user-guide/key-pair-auth)
- [Using Programmatic Access Tokens](https://docs.snowflake.com/en/user-guide/programmatic-access-tokens)
- [ALTER USER ... ROTATE PAT](https://docs.snowflake.com/en/sql-reference/sql/alter-user-rotate-programmatic-access-token)
- [ALTER USER ... ADD PAT](https://docs.snowflake.com/en/sql-reference/sql/alter-user-add-programmatic-access-token)
- [SHOW USER PROGRAMMATIC ACCESS TOKENS](https://docs.snowflake.com/en/sql-reference/sql/show-user-programmatic-access-tokens)
- [CREDENTIALS View (ACCOUNT_USAGE)](https://docs.snowflake.com/en/sql-reference/account-usage/credentials)
- [SYSTEM$DECODE_PAT](https://docs.snowflake.com/en/sql-reference/functions/system_decode_pat)
- [Authentication Policies](https://docs.snowflake.com/en/user-guide/authentication-policies)
- [Planning for Deprecation of Single-Factor Password Sign-ins](https://docs.snowflake.com/en/user-guide/security-mfa-rollout)
- [Snowflake Notebooks](https://docs.snowflake.com/en/user-guide/ui-snowsight/notebooks)

### AWS Documentation

- [Snowflake Key Pair (Secrets Manager Partner Integration)](https://docs.aws.amazon.com/secretsmanager/latest/userguide/mes-partner-Snowflake.html)
- [Rotation by Lambda Function](https://docs.aws.amazon.com/secretsmanager/latest/userguide/rotate-secrets_lambda.html)
- [Lambda Rotation Function Templates](https://docs.aws.amazon.com/secretsmanager/latest/userguide/reference_available-rotation-templates.html)
- [Permissions for Rotation](https://docs.aws.amazon.com/secretsmanager/latest/userguide/rotating-secrets-required-permissions-function.html)
- [Network Access for Lambda Rotation](https://docs.aws.amazon.com/secretsmanager/latest/userguide/rotation-function-network-access.html)